In [0]:
# Cell 1 — Create a Delta table
data = [
    (1, "RF", 88.0, "2026-03-01"),
    (2, "XGB", 92.0, "2026-03-02"),
    (3, "NN", 95.0, "2026-03-03")
]
df = spark.createDataFrame(
    data, ["id", "model", "accuracy", "date"]
)

# Write as Delta format using a managed table
df.write.format("delta").mode("overwrite").saveAsTable(
    "ml_experiments"
)
print("Delta table created!")

Delta table created!


In [0]:
# Cell 2 — Read it back
# Fix: Read from newly created managed table instead of /tmp/ml_experiments
delta_df = spark.read.format("delta").table("ml_experiments")
delta_df.show()

+---+-----+--------+----------+
| id|model|accuracy|      date|
+---+-----+--------+----------+
|  1|   RF|    88.0|2026-03-01|
|  2|  XGB|    92.0|2026-03-02|
|  3|   NN|    95.0|2026-03-03|
+---+-----+--------+----------+



In [0]:
# Cell 3 — UPDATE (Delta's superpower!)
from delta.tables import DeltaTable
# Use managed table instead of DBFS path
# DeltaTable.forName for managed tables

dt = DeltaTable.forName(spark, "ml_experiments")
dt.update(
    condition="model = 'RF'",
    set={"accuracy": "90.0"}
)
delta_df = spark.read.format("delta").table("ml_experiments")
delta_df.show()

+---+-----+--------+----------+
| id|model|accuracy|      date|
+---+-----+--------+----------+
|  2|  XGB|    92.0|2026-03-02|
|  3|   NN|    95.0|2026-03-03|
|  1|   RF|    90.0|2026-03-01|
+---+-----+--------+----------+



In [0]:
# Cell 4 — Time travel! (Delta's magic!)
spark.read.format("delta").option("versionAsOf", 0).table("ml_experiments").show()

+---+-----+--------+----------+
| id|model|accuracy|      date|
+---+-----+--------+----------+
|  1|   RF|    88.0|2026-03-01|
|  2|  XGB|    92.0|2026-03-02|
|  3|   NN|    95.0|2026-03-03|
+---+-----+--------+----------+

